<a href="https://colab.research.google.com/github/evinracher/3010090-ontological-engineering/blob/main/week5/workshop/part2/Taller_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Taller: Multi-Modal RAG con CLIP + Gemini (PDF con texto e imágenes)

Estructura (igual al taller de referencia):
- **Ejemplo** (código completo)
- **✍️ Tu turno** (espacio para completar)
- **❓ Preguntas** (qué pasaría si…)

**Objetivo:** Construir un mini Multi-Modal RAG que:
1) Extrae **texto e imágenes** desde un PDF  
2) Genera embeddings con **CLIP** (texto ↔ imagen en el mismo espacio)  
3) Indexa en **FAISS** con metadata (modalidad, página, etc.)  
4) Hace retrieval híbrido (texto+imagen) y responde con **Gemini** usando evidencia

## 1. Setup

In [ ]:
!pip -q install -U langchain_google_genai python-dotenv pymupdf tools langchain_community langchain langchain_core faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI


In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Extraer API KEY
try:
    from google.colab import userdata
    _k = userdata.get('GOOGLE_API_KEY')
    if _k:
        os.environ['GOOGLE_API_KEY'] = _k
except Exception:
    pass

# Inicializar el modelo Clip para embeddings unificados
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [ ]:
# Inicializar LLM

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0.2
)

print("modelo listo")

modelo listo


## 2. Crear funciones para generar embeddings usando CLIP

In [ ]:
def embed_image(image_data):
    """Crear embedding de imagen usando CLIP"""
    if isinstance(image_data, str):  # Si tenemos el path
        image = Image.open(image_data).convert("RGB")
    else:  # Si tenemos la imagen PIL
        image = image_data

    inputs=clip_processor(images=image,return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        # Normalizar embeddings al vector unitario
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

def embed_text(text):
    """Crear embedding de texto usando CLIP"""
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77  # CLIP max token length
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        # Normalizar embeddings
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

### ✍️ Tu turno

Prueba `embed_image` con una imagen artificial (sin depender del PDF).


In [ ]:
# TODO: crea una imagen simple con PIL (ej: un cuadrado blanco) y genera su embedding

### ❓ Preguntas
- ¿Qué pasa si pasas una imagen en escala de grises (modo "L")? ¿por qué el código hace `.convert("RGB")` en otros lugares?
- ¿Qué significa que CLIP produzca un vector en ℝ^d? (intuitivo: “coordenadas” semánticas)
- ¿Por qué embeddings visuales y textuales pueden compararse con coseno si viven en el mismo espacio?


## 3. Lectura del PDF a procesar

In [ ]:
# Procesar PDF
pdf_path="/content/1706.03762v7.pdf"
doc=fitz.open(pdf_path)

# Guardar todos los documentos y embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Guardar datos de la imagen para el LLM

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

### ✍️ Tu turno

Explora el PDF: páginas, texto e imágenes.


In [ ]:
# TODO: imprime número de páginas

# TODO: imprime un snippet del texto de la primera página

# TODO: cuenta imágenes por página (dict page->count) usando page.get_images(full=True)


### ❓ Preguntas
- ¿Qué pasa si el PDF es un escaneo? (texto vacío, pero imágenes sí)
- ¿Qué cambiarías para soportar PDFs muy grandes (100+ páginas) sin quedarte sin RAM?
- ¿En qué casos conviene chunking “layout-aware” en vez de chunking por caracteres?


## 4. Procesar con CLIP

In [ ]:
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def embed_text(text: str) -> np.ndarray:
    inputs = clip_processor(text=[text], images=None, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out = clip_model.get_text_features(**inputs)

    # Si devuelve un objeto, sacamos el tensor correcto
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        feats = out.pooler_output
    elif isinstance(out, (tuple, list)):
        feats = out[0]
    else:
        feats = out  # ya era tensor

    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.squeeze(0).detach().cpu().numpy()


@torch.no_grad()
def embed_image(pil_image) -> np.ndarray:
    inputs = clip_processor(text=None, images=pil_image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out = clip_model.get_image_features(**inputs)

    # Manejar salida tipo objeto
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        feats = out.pooler_output
    elif isinstance(out, (tuple, list)):
        feats = out[0]
    else:
        feats = out

    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.squeeze(0).detach().cpu().numpy()

### ✍️ Tu turno

Embebe 3 consultas en texto y compara similitud (coseno) entre ellas.


In [ ]:
# TODO: crea 3 queries sobre el PDF

# TODO: matriz de similitud coseno


### ❓ Preguntas
- Si dos queries son muy parecidas, ¿esperas sim más alto? ¿por qué?
- ¿Qué pasa si la query es muy larga? ¿qué hace `truncation=True`?
- ¿Por qué es común normalizar embeddings antes de comparar por coseno?


In [ ]:
for i,page in enumerate(doc):
    # Procesar texto
    text=page.get_text()
    if text.strip():
        # Crear documento temporal para splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        # Crear embedding de cada chunk usando CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)

    # Procesar imagenes
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convertir a imagen PIL
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            # Crear identificador único
            image_id = f"page_{i}_img_{img_index}"

            # Guardar imagenes en base64
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64

            # Crear embedding usando CLIP
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)

            # Crear documento para la imagen
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)

        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()

### ✍️ Tu turno

Verifica que se generaron documentos y embeddings, y revisa metadata.


In [ ]:
# TODO: imprime cuántos docs quedaron

# TODO: muestra 3 metadatas distintas


### ❓ Preguntas
- ¿Qué pasa si no guardas metadata como `type` y `page`? ¿cómo afecta filtros/explicabilidad?
- ¿Por qué en multimodal conviene guardar “evidencia” (ej: bytes de imagen) aparte (`image_data_store`)?
- ¿Cuál es el riesgo de chunking demasiado pequeño vs demasiado grande?


## 5. Usar FAISS para almacenar los vectores

In [ ]:
# Cree un almacén de vectores FAISS unificado con embeddings CLIP
embeddings_array = np.array(all_embeddings)
embeddings_array

array([[ 0.03500309,  0.00810928, -0.03526162, ...,  0.02110542,
         0.00167521, -0.03087724],
       [ 0.00678682, -0.00609661, -0.04333366, ..., -0.03352819,
        -0.0261889 , -0.01386963],
       [ 0.00037515, -0.02122335, -0.00457746, ..., -0.01093401,
         0.03803128, -0.02663146],
       ...,
       [-0.01654974, -0.04369025,  0.03837438, ..., -0.01301256,
         0.00746532, -0.07073215],
       [-0.01928525,  0.0009611 ,  0.00146912, ..., -0.03778007,
         0.02002625,  0.00973884],
       [-0.01122875, -0.02954051,  0.02860648, ..., -0.03015946,
         0.02209019,  0.00036673]], dtype=float32)

In [ ]:
# Crear un índice FAISS personalizado ya que tenemos embeddings precalculados
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

### ✍️ Tu turno

Haz una búsqueda simple y verifica que retorna items con metadata coherente.


In [ ]:
# TODO: corre una búsqueda por texto usando el vector_store

# TODO: imprime tipo y página de cada resultado


### ❓ Preguntas
- ¿Qué pasa si en vez de un índice unificado haces dos índices separados (texto e imagen)? ¿qué ganas y qué pierdes?
- ¿Qué síntomas ves cuando el retrieval trae “cerca” cosas irrelevantes? (y qué harías: rerank, filtros, mejor chunking)


## 6. Funciones para RAG multimodal

- Retrieve

In [ ]:
def retrieve_multimodal(query, k=5):
    """Retrieval unificado usando embeddings CLIP para texto e imagenes"""
    # Crear embedding de la query usando CLIP
    query_embedding = embed_text(query)

    # Buscar en almacén unificado
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )

    return results

### ✍️ Tu turno

Experimenta con `k` y observa cómo cambia el contexto recuperado.


In [ ]:
# TODO: prueba retrieve_multimodal con k=2 y k=8 y compara

# TODO: imprime (type, page) para cada set


### ❓ Preguntas
- ¿Qué pasa cuando k es muy grande? (ruido + tokens + costo)
- ¿Por qué a veces conviene “traer candidatos” con k grande y luego rerankear?
- ¿Cómo usarías metadata filtering (solo imágenes / solo página 1) si el vectorstore lo permite?


- Convertir imagen al formato aceptado por el llm

In [ ]:
import io, base64
from PIL import Image

def pil_to_data_url(pil_img: Image.Image, mime: str = "image/png") -> str:
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

- Crear el mensaje multimodal

In [ ]:
def create_multimodal_message(query, retrieved_docs):
    """Crear mensaje multimodal: texto + imagenes (como image_url) para ChatGoogleGenerativeAI."""
    parts = []

    # Separar documentos de texto e imagenes
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]

    # Construir contexto de texto
    parts.append(f"Question: {query}\n\nContext:\n")

    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata.get('page','?')}]: {doc.page_content}"
            for doc in text_docs
        ])
        parts.append(f"Text excerpts:\n{text_context}\n")

    # Recolectar imagenes como PIL
    images = []
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            try:
                img_bytes = base64.b64decode(image_data_store[image_id])
                img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                images.append(img)
                parts.append(f"\n[Image from page {doc.metadata.get('page','?')}]\n")
            except Exception as e:
                parts.append(f"\n[Image from page {doc.metadata.get('page','?')} could not be decoded: {e}]\n")

    parts.append("\n\nPlease answer the question based on the provided text and images.")
    prompt = "".join(parts)

    # Enviar imagenes como image_url (data URL)
    content = [{"type": "text", "text": prompt}]
    for img in images:
        data_url = pil_to_data_url(img, mime="image/png")
        content.append({"type": "image_url", "image_url": data_url})

    return HumanMessage(content=content)

### ✍️ Tu turno

Inspecciona el mensaje multimodal antes de enviarlo al LLM (estructura de `msg.content`).


In [ ]:
# TODO: crea un mensaje con 3 docs recuperados y mira su estructura

# TODO: imprime tipos de cada parte (text / image_url) y un preview corto


### ❓ Preguntas
- ¿Qué pasa si pasas demasiadas imágenes en un solo prompt?
- ¿Por qué es importante serializar la imagen como `data:image/png;base64,...`?
- ¿Qué diferencia hay entre “poner la imagen” y “describir la imagen”?


- Pipeline completo del RAG multimodal

In [ ]:
def invoke_multimodal(llm, msg: HumanMessage):
    try:
        return llm.invoke([msg])
    except ValueError as e:
        s = str(e)
        # Si la versión del llm usado no acepta image_url como string, lo convertimos a {"url": ...}
        if "Unrecognized message part type" in s or "image_url" in s:
            fixed = []
            for part in msg.content:
                if isinstance(part, dict) and part.get("type") == "image_url":
                    iu = part.get("image_url")
                    if isinstance(iu, str):
                        fixed.append({"type": "image_url", "image_url": {"url": iu}})
                    else:
                        fixed.append(part)
                else:
                    fixed.append(part)
            return llm.invoke([HumanMessage(content=fixed)])
        raise

In [ ]:
def multimodal_pdf_rag_pipeline(query):
    context_docs = retrieve_multimodal(query, k=5)
    msg = create_multimodal_message(query, context_docs)

    response = invoke_multimodal(llm, msg)
    return getattr(response, "content", str(response))

### ✍️ Tu turno

Crea una mini-evaluación: ejecuta 3 queries y guarda resultados en una tabla simple.


In [ ]:
if __name__ == "__main__":
    # Queries de ejemplo
    queries = [
        # Hacer preguntas sobre el PDF
    ]

    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: Why attention is important?
--------------------------------------------------
Answer: [{'type': 'text', 'text': "Based on the provided text, attention (specifically self-attention) is important for the following reasons:\n\n*   **Sequence Representation:** It allows for the relating of different positions within a single sequence to compute a more effective representation of that sequence.\n*   **Versatility in Tasks:** It has been used successfully across a variety of complex natural language processing tasks, including:\n    *   Reading comprehension\n    *   Abstractive summarization\n    *   Textual entailment\n    *   Learning task-independent sentence representations.\n*   **Performance Optimization:** The data on Page 8 suggests that adjusting attention parameters (such as the number of attention heads or the dimensionality of the keys/values) directly impacts the model's performance and accuracy (as seen in the varying BLEU scores).", 'extras': {'signature': 'EvwPCvkPA

### ❓ Preguntas
- ¿Qué pasa si el modelo “infiere” un valor de las imagenes pero no está en el contexto? ¿cómo lo detectas?
- ¿Qué señales indican que necesitas mejorar parsing de PDF?
- ¿Cómo harías “citas” más confiables?


- Pruebas

### ✍️ Tu turno

Modifica SOLO las queries (no el pipeline) para forzar retrieval de imágenes y luego de texto.


In [ ]:
# TODO: escribe 2 queries:
# 1) una que claramente apunte a un gráfico/figura (para traer imágenes)
# 2) una que apunte a definiciones o texto (para traer chunks textuales)
#
# Luego corre multimodal_pdf_rag_pipeline con ambas y compara qué te trajo el retrieval con retrieve_multimodal.


### ❓ Preguntas finales
- Si tu PDF tiene **tablas**, ¿qué estrategia usarías: convertir a Markdown, embedding por fila, o OCR? ¿por qué?
- ¿Cuándo Multi-Modal RAG NO vale la pena vs RAG textual?
